In [ ]:

import pandas as pd

# Load the datasets
train_data_path = 'D:\\LLM-Driven_AI-Studio\\MLAgent\\data\\benchmark\\DSEval\\datasets\\05_patient_profile\\train.csv'
test_data_path = 'D:\\LLM-Driven_AI-Studio\\MLAgent\\data\\benchmark\\DSEval\\datasets\\05_patient_profile\\test.csv'

train_df = pd.read_csv(train_data_path)
test_df = pd.read_csv(test_data_path)

# Display the first few rows of the training data
print(train_df.head())


                     Disease Fever  ... Cholesterol Level Outcome Variable
0                Ebola Virus   Yes  ...            Normal         Positive
1  Conjunctivitis (Pink Eye)    No  ...              High         Positive
2               Pancreatitis    No  ...              High         Positive
3               Pancreatitis   Yes  ...            Normal         Negative
4            Hyperthyroidism   Yes  ...            Normal         Negative

[5 rows x 10 columns]


In [ ]:

import pandas as pd

# Load the datasets
train_data_path = 'D:\\LLM-Driven_AI-Studio\\MLAgent\\data\\benchmark\\DSEval\\datasets\\05_patient_profile\\train.csv'
test_data_path = 'D:\\LLM-Driven_AI-Studio\\MLAgent\\data\\benchmark\\DSEval\\datasets\\05_patient_profile\\test.csv'

train_df = pd.read_csv(train_data_path)
test_df = pd.read_csv(test_data_path)

# One-hot encode the categorical features (except "Disease")
categorical_features = [col for col in train_df.columns if train_df[col].dtype == 'object' and col != 'Disease']
train_encoded = pd.get_dummies(train_df.drop('Disease', axis=1), columns=categorical_features)
test_encoded = pd.get_dummies(test_df.drop('Disease', axis=1), columns=categorical_features)

# Ensure both datasets have the same columns
test_encoded = test_encoded.reindex(columns=train_encoded.columns, fill_value=0)

# Display the first few rows of the encoded training data
print(train_encoded.head())


   Age  Fever_No  ...  Outcome Variable_Negative  Outcome Variable_Positive
0   50     False  ...                      False                       True
1   35      True  ...                      False                       True
2   35      True  ...                      False                       True
3   29     False  ...                       True                      False
4   35     False  ...                       True                      False

[5 rows x 19 columns]


In [ ]:


from sklearn.feature_selection import SelectKBest, chi2

# Define the features and target variable
X_train = train_encoded.drop('Outcome Variable_Negative', axis=1).drop('Outcome Variable_Positive', axis=1)
y_train = train_encoded['Outcome Variable_Negative']

# Perform feature selection
selector = SelectKBest(chi2, k=5)
X_new = selector.fit_transform(X_train, y_train)

# Get the selected feature names
selected_features = X_train.columns[selector.get_support()]

print(f"Selected features: {selected_features}")



Selected features: Index(['Blood Pressure_High', 'Blood Pressure_Low', 'Cholesterol Level_High',
       'Cholesterol Level_Low', 'Cholesterol Level_Normal'],
      dtype='object')


In [ ]:


from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Define the features and target variable
X_train_all = train_encoded.drop('Outcome Variable_Negative', axis=1)
y_train_all = train_encoded['Outcome Variable_Positive']

X_test_all = test_encoded.drop('Outcome Variable_Negative', axis=1)
y_test_all = test_encoded['Outcome Variable_Positive']

X_train_selected = train_encoded[selected_features]
X_test_selected = test_encoded[selected_features]

# Build and train Model1 with all one-hot encoded features
model1 = LogisticRegression(max_iter=1000)
model1.fit(X_train_all, y_train_all)
y_pred1 = model1.predict(X_test_all)

# Build and train Model2 with the selected features
model2 = LogisticRegression(max_iter=1000)
model2.fit(X_train_selected, y_train_all)
y_pred2 = model2.predict(X_test_selected)

# Evaluate Model1
accuracy1 = accuracy_score(y_test_all, y_pred1)
precision1 = precision_score(y_test_all, y_pred1)
recall1 = recall_score(y_test_all, y_pred1)
f1_1 = f1_score(y_test_all, y_pred1)

# Evaluate Model2
accuracy2 = accuracy_score(y_test_all, y_pred2)
precision2 = precision_score(y_test_all, y_pred2)
recall2 = recall_score(y_test_all, y_pred2)
f1_2 = f1_score(y_test_all, y_pred2)

# Print the evaluation metrics for both models
print(f"Model1 Performance:")
print(f"Accuracy: {accuracy1:.2f}, Precision: {precision1:.2f}, Recall: {recall1:.2f}, F1 Score: {f1_1:.2f}")

print("\nModel2 Performance:")
print(f"Accuracy: {accuracy2:.2f}, Precision: {precision2:.2f}, Recall: {recall2:.2f}, F1 Score: {f1_2:.2f}")




Model1 Performance:
Accuracy: 1.00, Precision: 1.00, Recall: 1.00, F1 Score: 1.00

Model2 Performance:
Accuracy: 0.61, Precision: 0.70, Recall: 0.57, F1 Score: 0.63
